# 🌦️ Analyse Exploratoire des Données Météo France
**Source : Météo-France – Données mensuelles par département (01 à 95)**  
Période couverte : 1950–2026 (deux fichiers par département : historique + récent)

---

## 1. Imports & configuration

In [ ]:
import os
import glob
import re
import warnings
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# --- Palette cohérente pour tous les graphiques ---
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ Librairies importées avec succès')

## 2. Chargement des fichiers (01 → 95)

Les fichiers suivent le nommage :  
- `MENSQ_{DD}_previous-1950-2024.csv`  
- `MENSQ_{DD}_latest-2025-2026.csv`  

Le numéro de département `{DD}` est extrait du nom de fichier **et** vérifié contre `NUM_POSTE`.

In [ ]:
# ── Adaptez ce chemin selon votre environnement ──
DATA_DIR = './data'   # répertoire contenant tous les CSV

# ── Patterns de nommage attendus ──
PATTERN_PREV   = 'MENSQ_{dd}_previous-*.csv'
PATTERN_LATEST = 'MENSQ_{dd}_latest-*.csv'

# ── Départements métropolitains 01–95 (hors 20 : Corse = 2A/2B) ──
DEPARTEMENTS = [f'{d:02d}' for d in range(1, 96) if d != 20]
# Ajout Corse si présents dans le dossier
for code_corse in ['2A', '2B']:
    if glob.glob(os.path.join(DATA_DIR, f'MENSQ_{code_corse}_previous-*.csv')):
        DEPARTEMENTS.append(code_corse)

print(f'Départements à charger : {len(DEPARTEMENTS)} codes')
print(DEPARTEMENTS[:10], '...')

In [ ]:
def load_dept(dept_code: str, data_dir: str) -> pd.DataFrame:
    """Charge et concatène les deux fichiers CSV d'un département."""
    frames = []
    patterns = [
        os.path.join(data_dir, f'MENSQ_{dept_code}_previous-*.csv'),
        os.path.join(data_dir, f'MENSQ_{dept_code}_latest-*.csv'),
    ]
    for pattern in patterns:
        files = glob.glob(pattern)
        for fpath in files:
            try:
                df = pd.read_csv(fpath, sep=None, engine='python', low_memory=False)
                df['_source_file'] = os.path.basename(fpath)
                frames.append(df)
            except Exception as e:
                print(f'  ⚠️  Erreur lecture {fpath} : {e}')
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def add_dept_column(df: pd.DataFrame, dept_code: str) -> pd.DataFrame:
    """Ajoute/vérifie la colonne DEP à partir du nom de fichier.
    NUM_POSTE : les 2 premiers chiffres (sur 8 digits) = code département.
    """
    if 'DEP' not in df.columns:
        df['DEP'] = dept_code
    else:
        # Vérification cohérence avec NUM_POSTE
        if 'NUM_POSTE' in df.columns:
            df['DEP_poste'] = df['NUM_POSTE'].astype(str).str.zfill(8).str[:2]
            incoherence = (df['DEP'] != df['DEP_poste']).sum()
            if incoherence:
                print(f'  ⚠️  {incoherence} lignes avec DEP incohérent pour le dept {dept_code}')
            df.drop(columns=['DEP_poste'], inplace=True)
    return df


# ── Chargement de tous les départements ──
all_frames = []
missing_depts = []

for dd in DEPARTEMENTS:
    df_dept = load_dept(dd, DATA_DIR)
    if df_dept.empty:
        missing_depts.append(dd)
        continue
    df_dept = add_dept_column(df_dept, dd)
    all_frames.append(df_dept)

if missing_depts:
    print(f'\n⚠️  Fichiers absents pour : {missing_depts}')

df_raw = pd.concat(all_frames, ignore_index=True)
print(f'\n✅ Dataset brut : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes')

## 3. Nettoyage & enrichissement temporel

In [ ]:
df = df_raw.copy()

# ── Colonne DATE ──────────────────────────────────────────────────────────────
# AAAAMM = entier YYYYMM  →  colonne datetime 'DATE' au 1er du mois
df['DATE'] = pd.to_datetime(df['AAAAMM'].astype(str), format='%Y%m', errors='coerce')
df['ANNEE'] = df['DATE'].dt.year
df['MOIS']  = df['DATE'].dt.month

# ── Tri chronologique ─────────────────────────────────────────────────────────
df.sort_values(['DEP', 'NUM_POSTE', 'DATE'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Doublons éventuels (même poste + même mois des deux fichiers) ─────────────
n_before = len(df)
df.drop_duplicates(subset=['NUM_POSTE', 'AAAAMM'], inplace=True)
n_after = len(df)
print(f'Doublons supprimés : {n_before - n_after}')
print(f'\nDataset final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Période : {df["DATE"].min().date()}  →  {df["DATE"].max().date()}')

## 4. Référentiel géographique : association Département → Région

Les données source ne contiennent pas la région. On télécharge le référentiel officiel COG (Code Officiel Géographique) de l'INSEE.

In [ ]:
# ── Référentiel INSEE : départements + régions ────────────────────────────────
# Source : https://www.data.gouv.fr/fr/datasets/departements-de-france/
REF_URL = (
    'https://raw.githubusercontent.com/gregoiredavid/france-geojson/'
    'master/departements-version-simplifiee.geojson'
)

# Table dept → région codée en dur (COG 2024 – régions métropolitaines)
DEPT_REGION = {
    '01':'Auvergne-Rhône-Alpes','02':'Hauts-de-France','03':'Auvergne-Rhône-Alpes',
    '04':'Provence-Alpes-Côte d\'Azur','05':'Provence-Alpes-Côte d\'Azur',
    '06':'Provence-Alpes-Côte d\'Azur','07':'Auvergne-Rhône-Alpes',
    '08':'Grand Est','09':'Occitanie','10':'Grand Est',
    '11':'Occitanie','12':'Occitanie','13':'Provence-Alpes-Côte d\'Azur',
    '14':'Normandie','15':'Auvergne-Rhône-Alpes','16':'Nouvelle-Aquitaine',
    '17':'Nouvelle-Aquitaine','18':'Centre-Val de Loire','19':'Nouvelle-Aquitaine',
    '2A':'Corse','2B':'Corse',
    '21':'Bourgogne-Franche-Comté','22':'Bretagne','23':'Nouvelle-Aquitaine',
    '24':'Nouvelle-Aquitaine','25':'Bourgogne-Franche-Comté','26':'Auvergne-Rhône-Alpes',
    '27':'Normandie','28':'Centre-Val de Loire','29':'Bretagne','30':'Occitanie',
    '31':'Occitanie','32':'Occitanie','33':'Nouvelle-Aquitaine','34':'Occitanie',
    '35':'Bretagne','36':'Centre-Val de Loire','37':'Centre-Val de Loire',
    '38':'Auvergne-Rhône-Alpes','39':'Bourgogne-Franche-Comté','40':'Nouvelle-Aquitaine',
    '41':'Centre-Val de Loire','42':'Auvergne-Rhône-Alpes','43':'Auvergne-Rhône-Alpes',
    '44':'Pays de la Loire','45':'Centre-Val de Loire','46':'Occitanie',
    '47':'Nouvelle-Aquitaine','48':'Occitanie','49':'Pays de la Loire',
    '50':'Normandie','51':'Grand Est','52':'Grand Est','53':'Pays de la Loire',
    '54':'Grand Est','55':'Grand Est','56':'Bretagne','57':'Grand Est',
    '58':'Bourgogne-Franche-Comté','59':'Hauts-de-France','60':'Hauts-de-France',
    '61':'Normandie','62':'Hauts-de-France','63':'Auvergne-Rhône-Alpes',
    '64':'Nouvelle-Aquitaine','65':'Occitanie','66':'Occitanie','67':'Grand Est',
    '68':'Grand Est','69':'Auvergne-Rhône-Alpes','70':'Bourgogne-Franche-Comté',
    '71':'Bourgogne-Franche-Comté','72':'Pays de la Loire','73':'Auvergne-Rhône-Alpes',
    '74':'Auvergne-Rhône-Alpes','75':'Île-de-France','76':'Normandie',
    '77':'Île-de-France','78':'Île-de-France','79':'Nouvelle-Aquitaine',
    '80':'Hauts-de-France','81':'Occitanie','82':'Occitanie',
    '83':'Provence-Alpes-Côte d\'Azur','84':'Provence-Alpes-Côte d\'Azur',
    '85':'Pays de la Loire','86':'Nouvelle-Aquitaine','87':'Nouvelle-Aquitaine',
    '88':'Grand Est','89':'Bourgogne-Franche-Comté','90':'Bourgogne-Franche-Comté',
    '91':'Île-de-France','92':'Île-de-France','93':'Île-de-France',
    '94':'Île-de-France','95':'Île-de-France',
}

ref_df = pd.DataFrame(list(DEPT_REGION.items()), columns=['DEP', 'REGION'])
df = df.merge(ref_df, on='DEP', how='left')

n_sans_region = df['REGION'].isna().sum()
if n_sans_region:
    print(f'⚠️  {n_sans_region} lignes sans région associée')
else:
    print('✅ Toutes les lignes ont une région')

df['REGION'].value_counts().to_frame('nb_lignes')

## 5. Vue d'ensemble du dataset fusionné

In [ ]:
print('=== SHAPE ===')
print(f'{df.shape[0]:,} lignes  |  {df.shape[1]} colonnes')
print(f'\nPériode   : {df["DATE"].min().date()} → {df["DATE"].max().date()}')
print(f'Depts     : {df["DEP"].nunique()} uniques')
print(f'Postes    : {df["NUM_POSTE"].nunique()} uniques')
print(f'Régions   : {df["REGION"].nunique()}')
print()
print('=== APERÇU DES COLONNES ===')
df.dtypes.reset_index().rename(columns={'index':'colonne', 0:'dtype'})

In [ ]:
# Aperçu des 5 premières lignes (colonnes clés)
cols_apercu = ['DEP','REGION','NUM_POSTE','NOM_USUEL','LAT','LON','ALTI','DATE','ANNEE','MOIS',
               'RR','TX','TN','TM','INST','ETP']
cols_apercu = [c for c in cols_apercu if c in df.columns]
df[cols_apercu].head(10)

In [ ]:
# Statistiques descriptives des variables numériques principales
vars_cles = ['RR', 'TX', 'TN', 'TM', 'INST', 'ETP', 'UNAB', 'FXIAB']
vars_cles = [v for v in vars_cles if v in df.columns]
df[vars_cles].describe().T.style.background_gradient(cmap='Blues', axis=1)

## 6. Qualité des données : valeurs manquantes

In [ ]:
# Taux de NaN par variable (variables numériques)
num_cols = df.select_dtypes(include='number').columns.tolist()
taux_nan = (df[num_cols].isna().mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
taux_nan[:40].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Taux de valeurs manquantes (%) – 40 premières variables', fontsize=14)
ax.set_ylabel('%')
ax.axhline(50, color='red', linestyle='--', linewidth=0.8, label='50 %')
ax.legend()
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

print(f'\nVariables avec > 80 % de NaN : {(taux_nan > 80).sum()}')
print(f'Variables avec > 50 % de NaN : {(taux_nan > 50).sum()}')
print(f'Variables complètes (0 % NaN) : {(taux_nan == 0).sum()}')

## 7. Points de collecte (bornes / postes) par département

In [ ]:
# Nombre de postes uniques par département
postes_dept = (
    df.groupby('DEP')['NUM_POSTE']
      .nunique()
      .reset_index()
      .rename(columns={'NUM_POSTE': 'nb_postes'})
      .sort_values('nb_postes', ascending=False)
)

moyenne_postes = postes_dept['nb_postes'].mean()
mediane_postes = postes_dept['nb_postes'].median()

print(f'Moyenne de postes par département : {moyenne_postes:.1f}')
print(f'Médiane de postes par département : {mediane_postes:.1f}')
print(f'Minimum : {postes_dept["nb_postes"].min()} ({postes_dept.iloc[-1]["DEP"]})')
print(f'Maximum : {postes_dept["nb_postes"].max()} ({postes_dept.iloc[0]["DEP"]})')

postes_dept.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
colors = ['tomato' if v > moyenne_postes else 'steelblue' for v in postes_dept['nb_postes']]
ax.bar(postes_dept['DEP'], postes_dept['nb_postes'], color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(moyenne_postes, color='black', linestyle='--', linewidth=1.2, label=f'Moyenne = {moyenne_postes:.1f}')
ax.set_title('Nombre de postes météo par département (rouge = au-dessus de la moyenne)', fontsize=13)
ax.set_xlabel('Département')
ax.set_ylabel('Nb postes')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Postes par région
postes_region = (
    df.groupby('REGION')['NUM_POSTE']
      .nunique()
      .reset_index()
      .rename(columns={'NUM_POSTE': 'nb_postes'})
      .sort_values('nb_postes', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(postes_region['REGION'], postes_region['nb_postes'], color='teal', edgecolor='white')
ax.set_title('Nombre total de postes météo par région', fontsize=13)
ax.set_xlabel('Nb postes')
plt.tight_layout()
plt.show()

## 8. Couverture temporelle par département

In [ ]:
cov_temp = df.groupby('DEP').agg(
    annee_min=('ANNEE', 'min'),
    annee_max=('ANNEE', 'max'),
    nb_mois=('DATE', 'nunique'),
    nb_postes=('NUM_POSTE', 'nunique'),
    nb_lignes=('NUM_POSTE', 'count'),
).reset_index()

cov_temp['amplitude_ans'] = cov_temp['annee_max'] - cov_temp['annee_min'] + 1

print('Statistiques de couverture temporelle :')
print(cov_temp[['DEP','annee_min','annee_max','amplitude_ans','nb_postes','nb_lignes']]
      .sort_values('annee_min').head(20))

## 9. Analyse des variables météo clés

### 9.1 Distribution des températures (TX, TN, TM)

In [ ]:
temp_vars = [v for v in ['TX', 'TN', 'TM'] if v in df.columns]
fig, axes = plt.subplots(1, len(temp_vars), figsize=(14, 4), sharey=False)

for ax, var in zip(axes, temp_vars):
    data = df[var].dropna()
    ax.hist(data, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.3,
               label=f'Moy. = {data.mean():.1f}°C')
    ax.set_title(var)
    ax.set_xlabel('°C')
    ax.legend(fontsize=9)

fig.suptitle('Distribution des températures mensuelles (toutes stations, toutes années)', fontsize=13)
plt.tight_layout()
plt.show()

### 9.2 Température moyenne annuelle nationale (TM)

In [ ]:
if 'TM' in df.columns:
    tm_annuel = df.groupby('ANNEE')['TM'].mean().reset_index()
    tm_annuel_valid = tm_annuel[tm_annuel['ANNEE'] >= 1950]

    # Tendance linéaire
    coeffs = np.polyfit(tm_annuel_valid['ANNEE'], tm_annuel_valid['TM'], 1)
    tendance = np.poly1d(coeffs)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(tm_annuel_valid['ANNEE'], tm_annuel_valid['TM'],
            color='steelblue', linewidth=1.5, label='TM annuelle moyenne')
    ax.plot(tm_annuel_valid['ANNEE'], tendance(tm_annuel_valid['ANNEE']),
            color='red', linestyle='--', linewidth=1.5,
            label=f'Tendance ({coeffs[0]:+.3f} °C/an)')
    ax.set_title('Évolution de la température mensuelle moyenne (TM) – France métropolitaine', fontsize=13)
    ax.set_xlabel('Année')
    ax.set_ylabel('°C')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'Tendance : {coeffs[0]:+.4f} °C / an  →  {coeffs[0]*30:+.2f} °C sur 30 ans')

### 9.3 Saisonnalité des précipitations (RR)

In [ ]:
if 'RR' in df.columns:
    rr_mois = df.groupby('MOIS')['RR'].mean().reset_index()
    mois_labels = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(rr_mois['MOIS'], rr_mois['RR'], color='teal', edgecolor='white')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(mois_labels)
    ax.set_title('Précipitations mensuelles moyennes (RR) – saisonnalité', fontsize=13)
    ax.set_ylabel('mm')
    plt.tight_layout()
    plt.show()

### 9.4 Comparaison régionale – Température et Précipitations

In [ ]:
vars_reg = [v for v in ['TM', 'RR', 'INST'] if v in df.columns]
reg_stats = df.groupby('REGION')[vars_reg].mean().reset_index()

for var in vars_reg:
    reg_sorted = reg_stats.sort_values(var, ascending=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(reg_sorted['REGION'], reg_sorted[var],
            color='coral' if var == 'TM' else 'steelblue' if var == 'RR' else 'gold',
            edgecolor='white')
    units = {'TM': '°C', 'RR': 'mm', 'INST': 'h'}
    noms  = {'TM': 'Température moyenne', 'RR': 'Précipitations', 'INST': 'Insolation'}
    ax.set_title(f'{noms.get(var, var)} par région (moyenne toutes années)', fontsize=13)
    ax.set_xlabel(units.get(var, ''))
    plt.tight_layout()
    plt.show()

### 9.5 Heatmap : corrélation entre variables clés

In [ ]:
vars_corr = [v for v in ['TX','TN','TM','RR','INST','ETP','UMM','FXIAB','UNAB'] if v in df.columns]
corr_matrix = df[vars_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Matrice de corrélation – variables météo clés', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Résumé & pistes d'analyse complémentaires

In [ ]:
print('=' * 60)
print('RÉSUMÉ DU DATASET FUSIONNÉ')
print('=' * 60)
print(f'  Lignes totales          : {len(df):,}')
print(f'  Colonnes                : {df.shape[1]}')
print(f'  Départements            : {df["DEP"].nunique()}')
print(f'  Régions                 : {df["REGION"].nunique()}')
print(f'  Postes de collecte      : {df["NUM_POSTE"].nunique()}')
print(f'  Moy. postes / dép.      : {postes_dept["nb_postes"].mean():.1f}')
print(f'  Période                 : {df["ANNEE"].min()} → {df["ANNEE"].max()}')
print()
print('Colonnes clés identifiées :')
print('  Température : TX, TN, TM, TMM')
print('  Précipitations : RR, RRAB, NBJRR1, …')
print('  Vent : FXIAB, FFM, FXI3SAB')
print('  Humidité : UMM, UNAB, UXAB')
print('  Ensoleillement : INST, GLOT, DIFT')
print('  Neige : HNEIGEFTOT, NBJNEIG, NEIGETOTM')
print('  Grêle / orage / brouillard : NBJGREL, NBJORAG, NBJBROU')
print()
print('Pistes d\'analyse complémentaires :')
print('  • Anomalie thermique par rapport à une période de référence (ex. 1981–2010)')
print('  • Détection des vagues de chaleur (NBJTX35, NBJTXI27)')
print('  • Évolution du nombre de jours de gel (NBJGELEE)')
print('  • Analyse de tendance par station (modèle linéaire ou Mann-Kendall)')
print('  • Cartographie sur fond GeoPandas (shapefiles IGN)')
print('  • Clustering de stations par profil climatique (k-means)')